In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

market = pd.read_csv("../data/market_multi_asset.csv")

market["Date"] = pd.to_datetime(market["Date"])
market = market.set_index("Date")

vix = yf.download("^INDIAVIX", start="2018-01-01", progress=False)

vix = vix.reset_index().set_index("Date")
vix = vix[["Close"]]
vix.columns = ["vix"]

usd = yf.download("USDINR=X", start="2018-01-01", progress=False)

usd = usd.reset_index().set_index("Date")
usd = usd[["Close"]]              # 🔴 IMPORTANT
usd.columns = ["usd_inr"]         # 🔴 FLATTEN

oil = yf.download("CL=F", start="2018-01-01", progress=False)

oil = oil.reset_index().set_index("Date")
oil = oil[["Close"]]              # 🔴 IMPORTANT
oil.columns = ["crude"]           # 🔴 FLATTEN

In [2]:
market = market.join(vix, how="left")
market = market.join(usd, how="left")
market = market.join(oil, how="left")

market = market.ffill()
market = market.dropna()

In [3]:
print(market.columns)

Index(['nifty_close', 'nifty_ret', 'bank_close', 'bank_ret',
       'relative_strength', 'breadth_signal', 'vix', 'usd_inr', 'crude'],
      dtype='object')


In [4]:
market["vix_ret"] = market["vix"].pct_change()
market["vix_signal"] = market["vix_ret"].rolling(5).mean()

market["usd_mom"] = market["usd_inr"].pct_change(10)
market["crude_mom"] = market["crude"].pct_change(10)

In [5]:
market = market.dropna()

In [6]:
market.to_csv("../data/market_master.csv")